In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import MDS, LocallyLinearEmbedding, TSNE
from src.visualization import plot_dim_reduction
from matplotlib.colors import ListedColormap
import pickle

In [0]:
# Load clustered results from gold layer
df_label = spark.table("scientific_trend_analysis.rep.arxiv_clustered").toPandas()
print(f"Loaded {len(df_label)} records with {len(df_label['Cluster_ID_km'].unique())} clusters")
print(f"Loaded {len(df_label)} records with {len(df_label['Cluster_ID_gmm_diag'].unique())} clusters")
print(f"Loaded {len(df_label)} records with {len(df_label['Cluster_ID_gmm_full'].unique())} clusters")
display(df_label.head())

#### K-Means Visuals

In [0]:
# Load LSA features from volume
with open('/Volumes/scientific_trend_analysis/core/model_artifacts/lsa_result_nor.pkl', 'rb') as f:
    lsa_result_nor = pickle.load(f)

# Extract cluster labels from the loaded dataframe
cluster_labels_km = df_label['Cluster_ID_km'].values

print(f"Loaded LSA features: {lsa_result_nor.shape}")

In [0]:
sample_fraction = 0.05
stratified_sample_km = df_label.groupby('Cluster_ID_km') \
                            .apply(lambda x: x.sample(frac=sample_fraction, random_state=2), include_groups=False)
sample_size = len(stratified_sample_km)

print(f'Sampled {sample_fraction*100}% of each cluster, total rows: {sample_size}\n')
print(df_label['Cluster_ID_km'].value_counts(normalize=True))

stratified_indices_km = stratified_sample_km.index.get_level_values(1).values
lsa_stratified_km = lsa_result_nor[stratified_indices_km]

# Stratified samples
cluster_labels_numpy_km = np.array(cluster_labels_km)
km_sample_labels = cluster_labels_numpy_km[stratified_indices_km]
print(f"\nStratified LSA features extracted for visualization: {lsa_stratified_km.shape}")

In [0]:
# Run MDS on stratified LSA sample
mds = MDS(
    n_components=2, 
    random_state=2, 
    n_init=1, 
    max_iter=300
)
mds_2d_km = mds.fit_transform(lsa_stratified_km)
df_plot_mds_km = plot_dim_reduction(mds_2d_km, km_sample_labels, 'MDS', 'Cluster_ID_km', sample_size, 'KMeans')

print(f"Stress: {mds.stress_:.2f}")
print(f"Converged in {mds.n_iter_} iterations")

In [0]:
# Try multiple n_neighbors values for LLE and select the best based on reconstruction error
n_neighbors_list = [5, 10, 15, 20]
lle_errors = {}
lle_results = {}

for n in n_neighbors_list:
    lle = LocallyLinearEmbedding(
        n_components=2, 
        random_state=2, 
        n_neighbors=n
    )
    lle_2d = lle.fit_transform(lsa_stratified_km)
    lle_errors[n] = lle.reconstruction_error_
    lle_results[n] = lle_2d
    print(f"n_neighbors={n}, Reconstruction Error: {lle.reconstruction_error_:.8f}")

best_n = min(lle_errors, key=lle_errors.get)
lle_2d_km = lle_results[best_n]
df_plot_lle_km = plot_dim_reduction(lle_2d_km, km_sample_labels, 'LLE', 'Cluster_ID_km', sample_size, 'KMeans')

print(f"Best n_neighbors: {best_n} with Reconstruction Error: {lle_errors[best_n]:.8f}")

In [0]:
# Use stratified sample for t-SNE visualization
tsne = TSNE(
    n_components=2, 
    random_state=2,
    perplexity=50,
    init='pca'
)
tsne_2d_km = tsne.fit_transform(lsa_stratified_km)
df_plot_tsne_km = plot_dim_reduction(tsne_2d_km, km_sample_labels, 't-SNE', 'Cluster_ID_km', sample_size, 'KMeans')

print(f"KL Divergence: {tsne.kl_divergence_:.4f}")

#### GMM Visuals - Part 1

In [0]:
# Load LSA features from volume
with open('/Volumes/scientific_trend_analysis/core/model_artifacts/lsa_result.pkl', 'rb') as f:
    lsa_result = pickle.load(f)

# Extract cluster labels from the loaded dataframe
cluster_labels_gmm_diag = df_label['Cluster_ID_gmm_diag'].values

print(f"Loaded LSA features: {lsa_result.shape}")

In [0]:
sample_fraction = 0.05
stratified_sample_gmm = df_label.groupby('Cluster_ID_gmm_diag') \
                            .apply(lambda x: x.sample(frac=sample_fraction, random_state=2), include_groups=False)
sample_size = len(stratified_sample_gmm)

print(f'Sampled {sample_fraction*100}% of each cluster, total rows: {sample_size}\n')
print(df_label['Cluster_ID_gmm_diag'].value_counts(normalize=True))

stratified_indices_gmm = stratified_sample_gmm.index.get_level_values(1).values
lsa_stratified_gmm = lsa_result[stratified_indices_gmm]

# Stratified samples
cluster_labels_numpy_gmm = np.array(cluster_labels_gmm_diag)
gmm_sample_labels = cluster_labels_numpy_gmm[stratified_indices_gmm]
print(f"\nStratified LSA features extracted for visualization: {lsa_stratified_gmm.shape}")

In [0]:
mds = MDS(
    n_components=2, 
    random_state=2, 
    n_init=1,
    max_iter=300
)
mds_2d_gmm = mds.fit_transform(lsa_stratified_gmm)
df_plot_mds_gmm = plot_dim_reduction(mds_2d_gmm, gmm_sample_labels, 'MDS', 'Cluster_ID_gmm_diag', sample_size, 'GaussianMixture')

print(f"Stress: {mds.stress_:.2f}")
print(f"Converged in {mds.n_iter_} iterations")

In [0]:
# Try multiple n_neighbors values for LLE and select the best based on reconstruction error
n_neighbors_list = [5, 10, 15, 20]
lle_errors = {}
lle_results = {}

for n in n_neighbors_list:
    lle = LocallyLinearEmbedding(
        n_components=2, 
        random_state=2, 
        n_neighbors=n
    )
    lle_2d = lle.fit_transform(lsa_stratified_gmm)
    lle_errors[n] = lle.reconstruction_error_
    lle_results[n] = lle_2d
    print(f"n_neighbors={n}, Reconstruction Error: {lle.reconstruction_error_:.8f}")

best_n = min(lle_errors, key=lle_errors.get)
lle_2d_gmm = lle_results[best_n]
df_plot_lle_gmm = plot_dim_reduction(lle_2d_gmm, gmm_sample_labels, 'LLE', 'Cluster_ID_gmm_diag', sample_size, 'GaussianMixture')

print(f"Best n_neighbors: {best_n} with Reconstruction Error: {lle_errors[best_n]:.8f}")

In [0]:
tsne = TSNE(
    n_components=2, 
    random_state=2,
    perplexity=50,
    learning_rate=200,
    early_exaggeration=12,
    init='pca'
)
tsne_2d_gmm = tsne.fit_transform(lsa_stratified_gmm)
df_plot_tsne_gmm = plot_dim_reduction(tsne_2d_gmm, gmm_sample_labels, 't-SNE', 'Cluster_ID_gmm_diag', sample_size, 'GaussianMixture')

print(f"KL Divergence: {tsne.kl_divergence_:.4f}")

#### GMM Visuals - Part 2

In [0]:
# Load LSA features from volume
with open('/Volumes/scientific_trend_analysis/core/model_artifacts/lsa_result_gmm_full.pkl', 'rb') as f:
    lsa_result_gmm_full = pickle.load(f)

# Extract cluster labels from the loaded dataframe
cluster_labels_gmm_full = df_label['Cluster_ID_gmm_full'].values

print(f"Loaded LSA features: {lsa_result_gmm_full.shape}")

In [0]:
sample_fraction = 0.05
stratified_sample_gmm = df_label.groupby('Cluster_ID_gmm_full') \
                            .apply(lambda x: x.sample(frac=sample_fraction, random_state=2), include_groups=False)
sample_size = len(stratified_sample_gmm)

print(f'Sampled {sample_fraction*100}% of each cluster, total rows: {sample_size}\n')
print(df_label['Cluster_ID_gmm_full'].value_counts(normalize=True))

stratified_indices_gmm = stratified_sample_gmm.index.get_level_values(1).values
lsa_stratified_gmm = lsa_result_gmm_full[stratified_indices_gmm]

# Stratified samples
cluster_labels_numpy_gmm = np.array(cluster_labels_gmm_full)
gmm_sample_labels = cluster_labels_numpy_gmm[stratified_indices_gmm]
print(f"\nStratified LSA features extracted for visualization: {lsa_stratified_gmm.shape}")

In [0]:
mds = MDS(
    n_components=2, 
    random_state=2, 
    n_init=1,
    max_iter=300
)
mds_2d_gmm = mds.fit_transform(lsa_stratified_gmm)
df_plot_mds_gmm = plot_dim_reduction(mds_2d_gmm, gmm_sample_labels, 'MDS', 'Cluster_ID_gmm_full', sample_size, 'GaussianMixture')

print(f"Stress: {mds.stress_:.2f}")
print(f"Converged in {mds.n_iter_} iterations")

In [0]:
# Try multiple n_neighbors values for LLE and select the best based on reconstruction error
n_neighbors_list = [5, 10, 15, 20]
lle_errors = {}
lle_results = {}

for n in n_neighbors_list:
    lle = LocallyLinearEmbedding(
        n_components=2, 
        random_state=2, 
        n_neighbors=n
    )
    lle_2d = lle.fit_transform(lsa_stratified_gmm)
    lle_errors[n] = lle.reconstruction_error_
    lle_results[n] = lle_2d
    print(f"n_neighbors={n}, Reconstruction Error: {lle.reconstruction_error_:.8f}")

best_n = min(lle_errors, key=lle_errors.get)
lle_2d_gmm = lle_results[best_n]
df_plot_lle_gmm = plot_dim_reduction(lle_2d_gmm, gmm_sample_labels, 'LLE', 'Cluster_ID_gmm_full', sample_size, 'GaussianMixture')

print(f"Best n_neighbors: {best_n} with Reconstruction Error: {lle_errors[best_n]:.8f}")

In [0]:
tsne = TSNE(
    n_components=2, 
    random_state=2,
    perplexity=50,
    learning_rate=200,
    early_exaggeration=12,
    init='pca'
)
tsne_2d_gmm = tsne.fit_transform(lsa_stratified_gmm)
df_plot_tsne_gmm = plot_dim_reduction(tsne_2d_gmm, gmm_sample_labels, 't-SNE', 'Cluster_ID_gmm_full', sample_size, 'GaussianMixture')

print(f"KL Divergence: {tsne.kl_divergence_:.4f}")

#### Temporal Analysis

In [0]:
# Use 12 distinct colors
color_list = plt.get_cmap('tab20').colors[:12]
cmap12 = ListedColormap(color_list)

# Create DataFrame with date and cluster
df_temporal = pd.DataFrame({
    'date': pd.to_datetime(df_label['update_date']),
    'cluster': cluster_labels_km
})

# Group by month and cluster
df_temporal['month'] = df_temporal['date'].dt.to_period('M')
temporal_trends = df_temporal.groupby(['month', 'cluster']).size().unstack(fill_value=0)

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Plot 1: Stacked area chart
temporal_trends.plot(kind='area', stacked=True, ax=axes[0], 
                     colormap=cmap12, alpha=0.85, linewidth=0)
# axes[0].set_title('Cluster Evolution Over Time (Stacked)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Year', fontsize=18)
axes[0].set_ylabel('Number of Papers', fontsize=18)
axes[0].tick_params(axis='x', labelsize=18)
axes[0].tick_params(axis='y', labelsize=18)
axes[0].legend(title='Cluster ID', bbox_to_anchor=(1.02, 1), loc='upper left')
axes[0].grid(True, alpha=0.5)
axes[0].set_ylim([0, 2000])

# Plot 2: Line chart (proportions)
temporal_props = temporal_trends.div(temporal_trends.sum(axis=1), axis=0)
temporal_props.plot(kind='line', ax=axes[1], colormap=cmap12, linewidth=2, alpha=0.85)
# axes[1].set_title('Cluster Proportions Over Time', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Year', fontsize=18)
axes[1].set_ylabel('Proportion', fontsize=18)
axes[1].tick_params(axis='x', labelsize=18)
axes[1].tick_params(axis='y', labelsize=18)
axes[1].legend(title='Cluster ID', bbox_to_anchor=(1.02, 1), loc='upper left')
axes[1].grid(True, alpha=0.5)
axes[1].set_ylim([0, 0.5])

plt.tight_layout()
plt.show()

# Identify trending clusters
recent_12m = temporal_trends.iloc[-12:]
growth = (recent_12m.iloc[-1] - recent_12m.iloc[0]) / (recent_12m.iloc[0] + 1)  # +1 to avoid division by zero
print("\n Cluster Growth (last 12 months):")
for cluster_id in growth.sort_values(ascending=False).index:
    print(f"  Cluster {cluster_id}: {growth[cluster_id]:+.1%}")

In [0]:
# Use 12 distinct colors
color_list = plt.get_cmap('tab20').colors[:12]
cmap12 = ListedColormap(color_list)

# Create DataFrame with date and cluster
df_temporal = pd.DataFrame({
    'date': pd.to_datetime(df_label['update_date']),
    'cluster': cluster_labels_gmm_full
})

# Group by month and cluster
df_temporal['month'] = df_temporal['date'].dt.to_period('M')
temporal_trends = df_temporal.groupby(['month', 'cluster']).size().unstack(fill_value=0)

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Plot 1: Stacked area chart
temporal_trends.plot(kind='area', stacked=True, ax=axes[0], 
                     colormap=cmap12, alpha=0.85, linewidth=0)
axes[0].set_title('Cluster Evolution Over Time (Stacked)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Year', fontsize=18)
axes[0].set_ylabel('Number of Papers', fontsize=18)
axes[0].tick_params(axis='x', labelsize=18)
axes[0].tick_params(axis='y', labelsize=18)
axes[0].legend(title='Cluster ID', bbox_to_anchor=(1.02, 1), loc='upper left')
axes[0].grid(True, alpha=0.5)
axes[0].set_ylim([0, 2000])

# Plot 2: Line chart (proportions)
temporal_props = temporal_trends.div(temporal_trends.sum(axis=1), axis=0)
temporal_props.plot(kind='line', ax=axes[1], colormap=cmap12, linewidth=2, alpha=0.85)
axes[1].set_title('Cluster Proportions Over Time', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Year', fontsize=18)
axes[1].set_ylabel('Proportion', fontsize=18)
axes[1].tick_params(axis='x', labelsize=18)
axes[1].tick_params(axis='y', labelsize=18)
axes[1].legend(title='Cluster ID', bbox_to_anchor=(1.02, 1), loc='upper left')
axes[1].grid(True, alpha=0.5)
axes[1].set_ylim([0, 0.5])

plt.tight_layout()
plt.show()

# Identify trending clusters
recent_12m = temporal_trends.iloc[-12:]
growth = (recent_12m.iloc[-1] - recent_12m.iloc[0]) / (recent_12m.iloc[0] + 1)  # +1 to avoid division by zero
print("\n Cluster Growth (last 12 months):")
for cluster_id in growth.sort_values(ascending=False).index:
    print(f"  Cluster {cluster_id}: {growth[cluster_id]:+.1%}")

#### Cluster Quality Evaluation Using Categories

In [0]:
df_label.head()

In [0]:
# Extract primary category prefix (e.g., 'cs.cv' -> 'cs', 'math.gr' -> 'math')
df_label['primary_category'] = df_label['categories'].str.split(' ').str[0].str.split('.').str[0]

# Create crosstab of cluster vs primary category
cluster_category_counts = pd.crosstab(df_label['Cluster_ID_km'], df_label['primary_category'])

# Get top categories across all clusters for better visualization
top_categories = df_label['primary_category'].value_counts().head(15).index
cluster_category_filtered = cluster_category_counts[top_categories]

# Create stacked bar plot
fig, ax = plt.subplots(figsize=(16, 8))
cluster_category_filtered.plot(kind='bar', stacked=True, ax=ax, 
                                colormap='tab20', alpha=0.85, width=0.8)

ax.set_title('Primary Category Distribution Across K-Means Clusters', fontsize=16, fontweight='bold')
ax.set_xlabel('Cluster ID', fontsize=14)
ax.set_ylabel('Number of Papers', fontsize=14)
ax.tick_params(axis='both', labelsize=12)
ax.legend(title='Primary Category', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=10)
ax.grid(True, alpha=0.3, axis='y')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# Overall cluster purity
total_dominant = cluster_category_counts.max(axis=1).sum()
total_papers = cluster_category_counts.sum().sum()
overall_purity = (total_dominant / total_papers) * 100
print(f"\nOverall Cluster Purity: {overall_purity:.1f}%")

# Show top 3 categories for each cluster
print("\n=== Top 3 Categories per Cluster ===")
for cluster_id in sorted(cluster_category_counts.index):
    top3 = cluster_category_counts.loc[cluster_id].nlargest(3)
    total = cluster_category_counts.loc[cluster_id].sum()
    print(f"\nCluster {cluster_id}:")
    for i, (cat, count) in enumerate(top3.items(), 1):
        pct = (count / total) * 100
        print(f"  {i}. {cat}: {count:,} papers ({pct:.1f}%)")

In [0]:
# Extract full category (first category before space, keep subcategory)
df_label['full_category'] = df_label['categories'].str.split(' ').str[0]

# Create crosstab of cluster vs full category
cluster_fullcat_counts = pd.crosstab(df_label['Cluster_ID_km'], df_label['full_category'])

# Get top categories across all clusters for better visualization
top_fullcategories = df_label['full_category'].value_counts().head(20).index
cluster_fullcat_filtered = cluster_fullcat_counts[top_fullcategories]

# Create stacked bar plot
fig, ax = plt.subplots(figsize=(18, 8))
cluster_fullcat_filtered.plot(kind='bar', stacked=True, ax=ax, 
                                colormap='tab20', alpha=0.85, width=0.8)

ax.set_title('Full Category Distribution Across K-Means Clusters', fontsize=16, fontweight='bold')
ax.set_xlabel('Cluster ID', fontsize=14)
ax.set_ylabel('Number of Papers', fontsize=14)
ax.tick_params(axis='both', labelsize=12)
ax.legend(title='Full Category', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
ax.grid(True, alpha=0.3, axis='y')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# Show top 5 full categories for each cluster
print("\n=== Top 5 Full Categories per Cluster ===")
for cluster_id in sorted(cluster_fullcat_counts.index):
    top5 = cluster_fullcat_counts.loc[cluster_id].nlargest(5)
    total = cluster_fullcat_counts.loc[cluster_id].sum()
    print(f"\nCluster {cluster_id}:")
    for i, (cat, count) in enumerate(top5.items(), 1):
        pct = (count / total) * 100
        print(f"  {i}. {cat}: {count:,} papers ({pct:.1f}%)")